# FREM-S14 GPU qualification and development

Default: all generated tests, 19-arm qualification, then 171 fixed historical proxy cells. The exact release ZIP is embedded. Attach borismarjanovic/price-volume-data-for-all-us-stocks-etfs. Select a GPU accelerator. 13500-second total limit. Set MODE to qualify for engineering only. Existing CUDA dependencies must satisfy the wheel requirements. Qualification is not a scientific result.


In [ ]:
import base64
import hashlib
import importlib.metadata
import io
import json
import os
import pathlib
import subprocess
import sys
import time
import zipfile

from packaging.requirements import Requirement
from packaging.version import Version

MODE = "historical_suite"
DEVICE = "cuda"
if MODE not in {"qualify", "historical_suite"}:
    raise ValueError("MODE must be qualify or historical_suite")
EXPECTED = "4651dc47f19bdb5bb91b00c8bd6900044d6118b0c31e2ba71c230f94fb52176f"
RELEASE = base64.b64decode(
    "...",
    validate=True,
)
if hashlib.sha256(RELEASE).hexdigest() != EXPECTED:
    raise RuntimeError("Embedded S14 release checksum mismatch")
TOTAL_DEADLINE = time.monotonic() + 13500
RUN = pathlib.Path("/kaggle/working") / (
    "frem-s14-" + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
)
RUN.mkdir(exist_ok=False)
(RUN / "release.zip").write_bytes(RELEASE)
PAYLOAD = RUN / "payload"
PAYLOAD.mkdir()
with zipfile.ZipFile(io.BytesIO(RELEASE)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError("duplicate archive members")
    inventory = json.loads(archive.read("PAYLOAD.json"))
    (PAYLOAD / "PAYLOAD.json").write_bytes(archive.read("PAYLOAD.json"))
    if set(names) != set(inventory) | {"PAYLOAD.json"}:
        raise ValueError("payload inventory mismatch")
    for name, expected in inventory.items():
        relative = pathlib.PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("unsafe archive member")
        data = archive.read(name)
        if len(data) != expected["bytes"] or hashlib.sha256(data).hexdigest() != expected["sha256"]:
            raise ValueError("payload hash mismatch")
        target = PAYLOAD / name
        target.parent.mkdir(exist_ok=True, parents=True)
        target.write_bytes(data)
INSTALL = RUN / "installed"
wheel = next(PAYLOAD.glob("*.whl"))
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--target",
        str(INSTALL),
        str(wheel),
    ],
    check=True,
)
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Python 3.11-3.13 required")
dist = next(importlib.metadata.distributions(path=[str(INSTALL)]))
for spec in dist.requires or []:
    requirement = Requirement(spec)
    if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
        continue
    version = importlib.metadata.version(requirement.name)
    if Version(version) not in requirement.specifier:
        raise RuntimeError(f"Incompatible installed dependency: {spec}; observed {version}")
ENV = dict(
    os.environ,
    PYTHONPATH=str(INSTALL),
    PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
    FREM_S14_TEST_DEVICE=DEVICE,
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)
(RUN / "delivery.json").write_text(
    json.dumps(
        {
            "archive_sha256": EXPECTED,
            "wheel_sha256": hashlib.sha256(wheel.read_bytes()).hexdigest(),
            "mode": MODE,
        },
        indent=2,
    )
)


def run(name, args, timeout):
    log = RUN / (name + ".log")
    started = time.monotonic()
    timeout = min(timeout, max(0, TOTAL_DEADLINE - started))
    if timeout <= 0:
        raise TimeoutError("13500-second notebook budget exhausted")
    with log.open("x") as stream:
        try:
            result = subprocess.run(
                [sys.executable, *args],
                env=ENV,
                cwd=PAYLOAD,
                stdout=stream,
                stderr=subprocess.STDOUT,
                timeout=timeout,
            )
            code = result.returncode
        except subprocess.TimeoutExpired:
            code = 124
    (RUN / (name + ".json")).write_text(
        json.dumps(
            {"returncode": code, "wall_seconds": time.monotonic() - started, "command": args},
            indent=2,
        )
    )
    print(log.read_text()[-16000:])
    if code:
        raise RuntimeError(f"{name} failed ({code}); retained {log}")


print("Verified release and isolated install:", RUN)

In [ ]:
run("preflight", ["-m", "frem_s14", "preflight", "--output", str(RUN / "host-admission.json")], 30)
run("tests", ["-m", "pytest", str(PAYLOAD / "tests"), "-W", "error", "-q"], 1800)
run(
    "qualify",
    [
        "-W",
        "error",
        "-m",
        "frem_s14",
        "qualify",
        "--device",
        DEVICE,
        "--output",
        str(RUN / "qualification"),
    ],
    1900,
)
run(
    "verify_qualification",
    ["-m", "frem_s14", "verify", "--bundle", str(RUN / "qualification")],
    300,
)

In [ ]:
if MODE == "historical_suite":
    run(
        "intake",
        ["-m", "frem_s14.intake", "--source", "/kaggle/input", "--output", str(RUN / "intake")],
        300,
    )
    run(
        "historical_suite",
        [
            "-W",
            "error",
            "-m",
            "frem_s14.suite",
            "--data",
            str(RUN / "intake/panel.csv"),
            "--qualification",
            str(RUN / "qualification"),
            "--output",
            str(RUN / "historical_suite"),
            "--device",
            DEVICE,
        ],
        11200,
    )
    run(
        "verify_suite",
        ["-m", "frem_s14", "verify-suite", "--bundle", str(RUN / "historical_suite")],
        1300,
    )
else:
    print("Generated qualification only; no market dataset opened.")
print("Retain/download all files under", RUN)